# Prepare data for EGA submission 

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
            
            # Seurat
            library(Seurat), 

            # Data 
            library(tidyverse), 
            library(data.table), 
            
            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/lab_stary/fdeckert/bin/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/lab_stary/fdeckert/FD20250213HSCT")

# Parameter 

In [ ]:
meta_select <- c(
    
    "patient_id", "time_point", 
    "celltype_low", "celltype_high", "celltype_domain", 
    "vireo_donor_id", "vireo_genotype_best_class", "genotype_class", 
    "v_a_gene", "j_a_gene", "cdr3_a_aa", "v_b_gene", "j_b_gene", "cdr3_b_aa", "clone_id", "clone_cell_n"
    
    )

# Seurat 

In [ ]:
so <- readRDS("data/object/pp.rds")

In [ ]:
so@meta.data <- so@meta.data[, meta_select]

In [ ]:
# saveRDS(so, "/research/lab_stary/fdeckert/FD20250213HSCT/data/zenodo/seurat.rds")

# AnnData

In [ ]:
# Store data as h5ad 
ad <- import("anndata", as="ad", convert=FALSE)
pd <- import("pandas", as="pd", convert=FALSE)
np <- import("numpy", as="np", convert=FALSE)
    
# Transform dgCMatrix to sparse sc_sparse matrix
adata <- ad$read_h5ad("data/object/pp.h5ad")
adata$obs <- adata$obs$filter(meta_select)

for (key in c("_scvi_manager_uuid", "_scvi_uuid", "", "", "digestion_protocol_colors", "leiden_colors", "leiden", "log1p", "msCC_label_colors", "solo_label_colors")) {adata$uns$pop(key, NULL)}

for (key in c("_scvi_extra_continuous_covs")) {adata$obsm$pop(key, NULL)}

adata$raw <- NULL

adata$write_h5ad("/research/lab_stary/fdeckert/FD20250213HSCT/data/zenodo/adata.h5ad")